# Entrega 2 · QLoRA para formular LP/MILP

**Orden de revisión:** datos → resultados medidos → carga del modelo → reproducción opcional de los 30 con ambas variantes → demostración del mismo caso → juicio posterior opcional → reproducción del ajuste. Requiere GPU T4 para inferencia. El benchmark oficial tiene 30 casos; el ejemplo oftalmológico de Entrega 1 se informa por separado. Las soluciones de referencia nunca se descargan en la sección de generación.

La media **0.5884** del baseline procede del CSV oficial congelado, no de una nueva ejecución. La demo al final hace **dos inferencias nuevas** y las guarda por separado. Los puntajes que muestra la tabla corresponden a los 31 juicios históricos/finales archivados, no a esas nuevas inferencias.


## 1. Instalación y archivos públicos

In [ ]:
%pip -q install transformers==4.51.3 peft==0.15.2 accelerate==1.6.0 bitsandbytes==0.45.5 datasets==3.5.0 pandas matplotlib


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv, hashlib, io, json
import pandas as pd

ROOT = Path('/content/deliverable_2')
ROOT.mkdir(exist_ok=True)
RAW = 'https://raw.githubusercontent.com/Jose21531/GenAI-Optimization/main/Deliverable%202/'
FILES = [
    'data/train_1050_user_only.jsonl',
    'data/benchmark_30_generation_only.jsonl',
    'data/caso_entrega1_generation_only.jsonl',
    'data/SHA256SUMS.generated.json',
    'outputs/baseline_31_respuestas_y_fom5.csv',
    'outputs/qlora_31_respuestas_y_fom5.csv',
]
for name in FILES:
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(urlopen(RAW + name).read())
expected = json.loads((ROOT/'data/SHA256SUMS.generated.json').read_text(encoding='utf-8'))
for name, digest in expected.items():
    actual = hashlib.sha256((ROOT/'data'/name).read_bytes()).hexdigest()
    assert actual == digest, name

def read_jsonl(name):
    return [json.loads(line) for line in (ROOT/'data'/name).read_text(encoding='utf-8').splitlines()]

train = read_jsonl('train_1050_user_only.jsonl')
benchmark = read_jsonl('benchmark_30_generation_only.jsonl')
case = read_jsonl('caso_entrega1_generation_only.jsonl')[0]
assert len(train) == 1050 and len(benchmark) == 30 and case['id'] == 'diagnostic_oftalmo'
assert all('reference_model' not in item and 'requirements' not in item for item in benchmark)
assert all([m['role'] for m in item['messages']] == ['user','assistant'] for item in train)
print('Datos: 1050 ejemplos de entrenamiento, 30 entradas de benchmark y 1 caso diagnóstico.')
print('Diversidad:', len({x['family_id'] for x in train}), 'familias;',
      sum(x['model_type']=='LP' for x in train), 'LP y',
      sum(x['model_type']=='MILP' for x in train), 'MILP.')


## 2. Resultados oficiales y comparación pareada

Cada CSV contiene la respuesta completa del SLM y el juicio FOM-5 v2. El baseline se recuperó de `resultados_fom5_local.csv` y QLoRA se generó después del ajuste final. El juez Qwen3-14B se aplicó **después** de generar, con la referencia reservada al evaluador. El intervalo bootstrap se calcula aquí sobre diferencias pareadas; no lo entrega automáticamente el juez.


In [ ]:
base = pd.read_csv(ROOT/'outputs/baseline_31_respuestas_y_fom5.csv')
adapt = pd.read_csv(ROOT/'outputs/qlora_31_respuestas_y_fom5.csv')
b = base.query('split == "benchmark"').set_index('id').sort_index()
a = adapt.query('split == "benchmark"').set_index('id').sort_index()
assert len(b) == len(a) == 30 and b.index.equals(a.index)
paired = pd.DataFrame({'baseline': b.score_fom5_v2, 'QLoRA': a.score_fom5_v2})
paired['cambio'] = paired.QLoRA - paired.baseline
counts = {'mejoran': int((paired.cambio > 0).sum()),
          'sin cambio': int((paired.cambio == 0).sum()),
          'empeoran': int((paired.cambio < 0).sum())}
assert counts == {'mejoran': 14, 'sin cambio': 8, 'empeoran': 8}
assert abs(paired.baseline.mean() - .5884) < 1e-10
assert abs(paired.QLoRA.mean() - .8023666666666667) < 1e-10
print(f"Media 30: {paired.baseline.mean():.4f} → {paired.QLoRA.mean():.4f}/5; cambio {paired.cambio.mean():+.4f}")
print('Por caso:', counts)
import numpy as np
rng = np.random.default_rng(42)
means = rng.choice(paired.cambio.to_numpy(), size=(10000, 30), replace=True).mean(axis=1)
print('IC bootstrap 95% del cambio pareado:', np.quantile(means, [.025, .975]).round(4))
print('Caso Entrega 1, aparte: baseline determinista',
      float(base.query('id == "diagnostic_oftalmo"').score_fom5_v2.iloc[0]),
      '→ QLoRA', float(adapt.query('id == "diagnostic_oftalmo"').score_fom5_v2.iloc[0]))
display(paired.sort_values('cambio', ascending=False))


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(['Baseline','QLoRA'], [paired.baseline.mean(), paired.QLoRA.mean()], color=['#74899b','#087f8c'])
axes[0].set_ylim(0, 5); axes[0].set_ylabel('FOM-5 v2 / 5'); axes[0].set_title('Media de 30 casos')
axes[1].bar(counts.keys(), counts.values(), color=['#087f8c','#74899b','#b85b49'])
axes[1].set_ylim(0, 30); axes[1].set_ylabel('Casos'); axes[1].set_title('Cambio por problema')
plt.tight_layout(); plt.show()


## 3. Preparar el SLM y el adapter público

El adapter final se descarga del repositorio con verificación SHA-256; no hace falta la cuenta ni el Drive del equipo. La misma instancia de Qwen se usa con el adapter apagado para el baseline y encendido para QLoRA. La generación es determinista, user-only y tiene el mismo límite de 1200 tokens.


In [ ]:
from zipfile import ZipFile
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

assert torch.cuda.is_available(), 'Selecciona Entorno de ejecución → Cambiar tipo → T4 GPU.'
archive = ROOT/'model/adapter_qlora_qwen25_final.zip'
archive.parent.mkdir(parents=True, exist_ok=True)
if not archive.exists():
    archive.write_bytes(urlopen(RAW+'model/adapter_qlora_qwen25_final.zip').read())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '68583e9dffa6647a5c0b89a74168363264daf246e4337ef1ae0c644243391151'
ADAPTER = ROOT/'model/adapter'
ADAPTER.mkdir(exist_ok=True)
with ZipFile(archive) as zipped:
    assert set(zipped.namelist()) == {'adapter_config.json','adapter_model.safetensors'}
    zipped.extractall(ADAPTER)
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
set_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, torch_dtype=torch.float16, device_map={'': 0})
model = PeftModel.from_pretrained(base_model, ADAPTER, is_trainable=False)
model.eval()
print('T4 y adapter público preparados.')


## 4. Reproducir las 30 entradas con ambas variantes (opcional)

Esta celda **sí ejecuta el SLM** sobre los 30 enunciados oficiales sin gold: primero 30 baseline y después los mismos 30 QLoRA. Guarda una línea por ID inmediatamente y reanuda sin repetir las ya escritas. Activa `RUN_REGENERATE_30=True` solo cuando quieras gastar el tiempo de GPU (60 generaciones; puede tomar bastante más que la demo). Estos archivos son **nuevas inferencias sin puntaje**; los CSV oficiales de la sección 2 permanecen intactos. El juicio FOM-5 v2 de nuevas respuestas es un paso separado posterior.


In [ ]:
RUN_REGENERATE_30 = False  # True: 30 baseline y luego las mismas 30 QLoRA.
if RUN_REGENERATE_30:
    import time
    from tqdm.auto import tqdm
    assert len(benchmark) == 30 and len({r['id'] for r in benchmark}) == 30
    new_dir = ROOT/'outputs/reproduction_30'
    new_dir.mkdir(parents=True, exist_ok=True)
    def load_completed(path):
        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()] if path.exists() else []
        assert len(rows) == len({row['id'] for row in rows}), path
        assert {row['id'] for row in rows} <= {item['id'] for item in benchmark}
        expected = {item['id']:hashlib.sha256(item['problem'].encode()).hexdigest() for item in benchmark}
        assert all(row['problem_sha256']==expected[row['id']] and row['variant']==path.stem
                   and row['candidate'].strip() for row in rows), path
        return rows
    def generate_benchmark(item):
        prompt = tokenizer.apply_chat_template([{'role':'user','content':item['problem']}],
                                               tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
        torch.cuda.synchronize(); started = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=1200, do_sample=False,
                                    pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        new = output[0, inputs['input_ids'].shape[1]:]
        return {'id':item['id'], 'title':item['title'], 'split':'benchmark',
                'model':MODEL, 'problem_sha256':hashlib.sha256(item['problem'].encode()).hexdigest(),
                'candidate':tokenizer.decode(new, skip_special_tokens=True).strip(),
                'prompt_tokens':int(inputs['input_ids'].shape[1]),
                'output_tokens':int(new.numel()), 'latency_sec':round(time.perf_counter()-started,3),
                'reached_token_budget':bool(new.numel() >= 1200)}
    for label in ('baseline_direct_new_run', 'qlora_new_run'):
        path = new_dir/f'{label}.jsonl'
        done = {row['id'] for row in load_completed(path)}
        print('Iniciando',label,'faltantes',30-len(done),flush=True)
        def run_missing():
            for item in tqdm(benchmark, desc=label):
                if item['id'] in done: continue
                row = generate_benchmark(item)
                row['variant'] = label
                assert row['candidate']
                with path.open('a',encoding='utf-8') as stream:
                    stream.write(json.dumps(row,ensure_ascii=False)+'\n')
                done.add(item['id'])
                print(item['id'],row['output_tokens'],'tokens',row['latency_sec'],'s',flush=True)
        if label == 'baseline_direct_new_run':
            with model.disable_adapter(): run_missing()
        else:
            run_missing()
        rows = load_completed(path)
        assert len(rows) == 30
        pd.DataFrame(rows).to_csv(new_dir/f'{label}.csv',index=False,encoding='utf-8-sig')
    print('Dos CSV nuevos sin score:',new_dir)
else:
    print('Reproducción de 30 desactivada: se muestran los dos CSV oficiales ya evaluados en la sección 2.')


## 5. Demostración en vivo: mismo caso de Entrega 1

Inicia la grabación antes de ejecutar la próxima celda. El baseline se genera con el adapter apagado y QLoRA con el adapter encendido, sobre la misma entrada. Ambos textos nuevos se guardan en `/content/deliverable_2/outputs/live_demo.json`; no se les asignan los puntajes archivados.


In [ ]:
import time
print('MISMA ENTRADA · CENTROS OFTALMOLÓGICOS (Entrega 1)')
print(case['problem'])
prompt = tokenizer.apply_chat_template([{'role':'user','content':case['problem']}],
                                       tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)

def generate_now():
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        tokens = model.generate(**inputs, max_new_tokens=1200, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    new = tokens[0, inputs['input_ids'].shape[1]:]
    return {'candidate': tokenizer.decode(new, skip_special_tokens=True).strip(),
            'output_tokens': int(new.numel()), 'latency_sec': round(time.perf_counter()-started, 3)}

print('Generando baseline directo ahora...', flush=True)
with model.disable_adapter():
    live_base = generate_now()
print('Baseline listo:', live_base['output_tokens'], 'tokens', flush=True)
print('Generando QLoRA ahora...', flush=True)
live_qlora = generate_now()
print('QLoRA listo:', live_qlora['output_tokens'], 'tokens', flush=True)

demo_path = ROOT / 'outputs/live_demo.json'
demo_path.parent.mkdir(parents=True, exist_ok=True)
live_demo = {'id': case['id'], 'problem': case['problem'],
             'problem_sha256': hashlib.sha256(case['problem'].encode()).hexdigest(),
             'model_revision': REVISION, 'prompt': 'user_only', 'do_sample': False,
             'max_new_tokens': 1200, 'baseline_new_run': live_base,
             'qlora_new_run': live_qlora}
demo_path.write_text(json.dumps(live_demo, ensure_ascii=False, indent=2), encoding='utf-8')
print('Dos respuestas nuevas guardadas en:', demo_path)


### Respuestas de la demostración, lado a lado

In [ ]:
from IPython.display import HTML, display
from html import escape
cards = []
for label, run, color in [('Baseline directo · nueva ejecución', live_base, '#74899b'),
                          ('QLoRA · nueva ejecución', live_qlora, '#087f8c')]:
    cards.append(f'''<div style="flex:1;min-width:290px;border:2px solid {color};border-radius:8px;padding:12px">
      <h3 style="color:{color};margin-top:0">{label}</h3>
      <p>{run['output_tokens']} tokens · {run['latency_sec']:.1f} s</p>
      <pre style="white-space:pre-wrap;max-height:540px;overflow:auto;font-size:12px">{escape(run['candidate'])}</pre></div>''')
display(HTML('<div style="display:flex;gap:12px">' + ''.join(cards) + '</div>'))
print('Puntajes del juicio archivado (no de esta ejecución): 0.650 → 3.850 / 5.')
print('Advertencia matemática: QLoRA aún omite p_j en el objetivo del caso oftalmológico.')


## 6. Evaluar las 60 respuestas nuevas con FOM-5 v2 (opcional, posterior a la demo)

Si activaste la sección 4 y tienes **30 respuestas completas de cada variante**, ejecuta esta sección para obtener dos CSV nuevos con texto y puntaje. Descarga gold y requisitos **solo ahora**, después de generar. Libera el SLM de la GPU, carga Qwen3-14B NF4 en la misma revisión usada originalmente, calibra cinco casos y aplica el prompt, parser y agregación del evaluador autoritativo. Es una segunda ejecución costosa; los resultados oficiales de la sección 2 siguen congelados y no se sustituyen. Los archivos nuevos se guardan en `outputs/reproduction_30/`.


In [ ]:
RUN_JUDGE_NEW_30 = False  # True solo tras completar ambos JSONL de la sección 4.
if RUN_JUDGE_NEW_30:
    import gc, re, time
    from transformers import BitsAndBytesConfig
    from tqdm.auto import tqdm
    new_dir = ROOT/'outputs/reproduction_30'
    for name in ('baseline_direct_new_run','qlora_new_run'):
        path = new_dir/f'{name}.jsonl'
        assert path.exists() and len(path.read_text(encoding='utf-8').splitlines()) == 30, path
    EVAL_FILES = [
        'data/source/benchmark_30_v2.jsonl',
        'evidence/caso_entrega1_evaluation_only.json',
        'evidence/resultados_fom5_local_original.csv',
        'evidence/evaluation_manifest.json',
    ]
    for name in EVAL_FILES:
        path = ROOT/name
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(urlopen(RAW+name).read())
    manifest = json.loads((ROOT/'evidence/evaluation_manifest.json').read_text(encoding='utf-8'))
    assert hashlib.sha256((ROOT/'data/source/benchmark_30_v2.jsonl').read_bytes()).hexdigest() == manifest['gold_sha256']
    assert hashlib.sha256((ROOT/'evidence/caso_entrega1_evaluation_only.json').read_bytes()).hexdigest() == manifest['diagnostic_gold_sha256']
    GOLD = [json.loads(line) for line in (ROOT/'data/source/benchmark_30_v2.jsonl').read_text(encoding='utf-8').splitlines()]
    GOLD.append(json.loads((ROOT/'evidence/caso_entrega1_evaluation_only.json').read_text(encoding='utf-8')))
    GOLD_LOOKUP = {row['id']:row for row in GOLD}
    assert len(GOLD_LOOKUP)==31
    old_df = pd.read_csv(ROOT/'evidence/resultados_fom5_local_original.csv')
    del model, base_model
    gc.collect(); torch.cuda.empty_cache()
    JUDGE_MODEL = 'Qwen/Qwen3-14B'
    quant = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL,revision=manifest['judge_revision'])
    model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL,revision=manifest['judge_revision'],
        quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,low_cpu_mem_usage=True)
    model.eval()
    print('Juez FOM-5 v2 preparado. Gold cargado solo después de generación.')
else:
    print('Juez nuevo desactivado; los 31 juicios originales y la comparación están en la sección 2.')


In [ ]:
SYSTEM_JUDGE = r"""
Eres un juez estricto de formulaciones de Investigación de Operaciones.

TAREA DEL CANDIDATO:
Traducir un problema PL/MILP desde lenguaje natural a una formulación matemática.
Debe formular, NO resolver numéricamente.

REGLAS:
1. Evalúa solo lo escrito. No completes ecuaciones faltantes.
2. Acepta formulaciones matemáticamente equivalentes al gold aunque cambien símbolos o técnica.
3. Una restricción esencial omitida, invertida o aplicada a la variable equivocada es missing/wrong.
4. Una explicación, algoritmo, código o solución numérica no sustituye una formulación matemática.
5. Si falta una variable decisional esencial, decision_domains <= 2.
6. Si el objetivo tiene el sentido correcto pero omite un término esencial, objective <= 2.
7. Si no hay objetivo matemático, objective <= 1.
8. Si no existe formulación matemática identificable, has_formulation=false.
9. Contradicciones entre ecuaciones penalizan constraints_logic y algebra_validity.
10. Si formula correctamente pero además resuelve la instancia, generalization <= 4.
11. Si sustituye la formulación por una solución numérica, has_formulation=false.
12. No premies prosa ni castigues notación distinta si es coherente.

ESCALA 0-5:
0 ausente/totalmente inválido
1 falla fundamental
2 estructura reconocible con error esencial
3 mayormente correcta, pero falta/falla una pieza material
4 correcta salvo detalle menor
5 completa y matemáticamente correcta/equivalente

Estados de requisito permitidos:
met, partial, missing, wrong

Devuelve SOLO JSON válido. Sin markdown y sin explicaciones largas.
"""

DIMENSIONS = [
    "decision_domains",
    "objective",
    "constraints_logic",
    "algebra_validity",
    "generalization",
]

def judge_prompt(item, candidate):
    req_lines = "\n".join(
        f'{r["id"]} [{r["kind"]}]: {r["text"]}'
        for r in item["requirements"]
    )

    req_template = ", ".join(
        f'"{r["id"]}":"met|partial|missing|wrong"'
        for r in item["requirements"]
    )

    return f"""
ENUNCIADO:
{item["problem"]}

REQUISITOS:
{req_lines}

GOLD DE REFERENCIA:
{item["reference_model"]}

RESPUESTA CANDIDATA:
{candidate}

Responde exactamente con esta estructura:
{{
  "has_formulation": true,
  "requirements": {{{req_template}}},
  "scores": {{
    "decision_domains": 0,
    "objective": 0,
    "constraints_logic": 0,
    "algebra_validity": 0,
    "generalization": 0
  }},
  "fatal_errors": ["error breve si corresponde"]
}}
"""

MAX_INPUT_TOKENS = 12000
MAX_NEW_TOKENS = 600

def _chat_text(messages):
    # Qwen3 permite desactivar thinking explícito.
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def extract_json(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    start = text.find("{")
    if start < 0:
        raise ValueError("No se encontró inicio de JSON.")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i+1])

    raise ValueError("JSON incompleto.")

def validate_parsed(item, parsed):
    if not isinstance(parsed, dict):
        raise ValueError("La respuesta parseada no es un objeto.")

    if "has_formulation" not in parsed:
        raise ValueError("Falta has_formulation.")

    reqs = parsed.get("requirements")
    if not isinstance(reqs, dict):
        raise ValueError("requirements debe ser un objeto {R1: estado, ...}.")

    expected = {r["id"] for r in item["requirements"]}
    got = set(reqs)

    if expected != got:
        raise ValueError(
            f"IDs de requisitos incorrectos. Esperados={expected}; recibidos={got}"
        )

    allowed = {"met", "partial", "missing", "wrong"}
    bad_status = {
        rid: status
        for rid, status in reqs.items()
        if str(status).lower().strip() not in allowed
    }
    if bad_status:
        raise ValueError(f"Estados inválidos: {bad_status}")

    scores = parsed.get("scores")
    if not isinstance(scores, dict):
        raise ValueError("Falta scores.")

    for dim in DIMENSIONS:
        if dim not in scores:
            raise ValueError(f"Falta score: {dim}")
        value = float(scores[dim])
        if not (0 <= value <= 5):
            raise ValueError(f"Score fuera de rango en {dim}: {value}")

    if "fatal_errors" not in parsed:
        parsed["fatal_errors"] = []

    return parsed

@torch.inference_mode()
def judge_local(item, candidate, max_retries=2):
    prompt = judge_prompt(item, candidate)
    messages = [
        {"role": "system", "content": SYSTEM_JUDGE},
        {"role": "user", "content": prompt},
    ]

    last_error = None
    last_raw = None

    for attempt in range(max_retries):
        local_messages = list(messages)
        if attempt > 0:
            local_messages.append({
                "role": "user",
                "content": (
                    "La salida anterior no pudo parsearse. "
                    "Repite el mismo juicio y devuelve SOLO el JSON solicitado, "
                    "completo y sin texto adicional."
                ),
            })

        text = _chat_text(local_messages)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

        n_input = int(inputs["input_ids"].shape[1])
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(
                f"Prompt de {n_input} tokens supera MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}. "
                "No se truncó silenciosamente."
            )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        try:
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            generated = output[0, inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            last_raw = raw

            parsed = validate_parsed(item, extract_json(raw))
            return parsed, raw, n_input, int(generated.shape[0])

        except Exception as e:
            last_error = str(e)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"Judge local falló tras {max_retries} intentos: {last_error}\n"
        f"Última salida: {last_raw}"
    )

STATUS_VALUE = {
    "met": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "wrong": 0.0,
}

def safe_score(value):
    return max(0.0, min(5.0, float(value)))

def aggregate_v2(item, parsed):
    status = {
        rid: STATUS_VALUE[str(st).lower().strip()]
        for rid, st in parsed["requirements"].items()
    }

    by_kind = {"objective": [], "constraint": [], "domain": []}

    for req in item["requirements"]:
        by_kind[req["kind"]].append(
            status.get(req["id"], 0.0)
        )

    obj_cov = np.mean(by_kind["objective"]) if by_kind["objective"] else 1.0
    con_cov = np.mean(by_kind["constraint"]) if by_kind["constraint"] else 1.0
    dom_cov = np.mean(by_kind["domain"]) if by_kind["domain"] else 1.0

    raw = parsed["scores"]

    decision = safe_score(raw["decision_domains"])
    objective = min(safe_score(raw["objective"]), 5.0 * obj_cov)
    constraints = min(safe_score(raw["constraints_logic"]), 5.0 * con_cov)
    algebra = safe_score(raw["algebra_validity"])
    generalization = safe_score(raw["generalization"])

    if by_kind["domain"]:
        decision = min(decision, 5.0 * dom_cov)

    score = (
        0.15 * decision
        + 0.20 * objective
        + 0.35 * constraints
        + 0.15 * algebra
        + 0.15 * generalization
    )

    if not bool(parsed["has_formulation"]):
        algebra = min(algebra, 1.0)
        generalization = min(generalization, 1.0)

        score = min(
            1.5,
            0.15 * decision
            + 0.20 * objective
            + 0.35 * constraints
            + 0.15 * algebra
            + 0.15 * generalization
        )

    all_cov = np.mean(list(status.values())) if status else np.nan

    return {
        "score_fom5_v2": round(float(score), 3),
        "decision_domains_v2": float(decision),
        "objective_v2": float(objective),
        "constraints_logic_v2": float(constraints),
        "algebra_validity_v2": float(algebra),
        "generalization_v2": float(generalization),
        "requirement_coverage_v2": float(all_cov),
        "objective_coverage": float(obj_cov),
        "constraint_coverage": float(con_cov),
        "domain_coverage": float(dom_cov),
    }



In [ ]:
if RUN_JUDGE_NEW_30:
    calibration_spec = [
        ('opt_02','baseline_direct_deterministic',2.8),
        ('opt_10','baseline_direct_deterministic',1.5),
        ('opt_14','baseline_direct_deterministic',1.5),
        ('opt_16','baseline_direct_deterministic',1.8),
        ('diagnostic_oftalmo','entrega1_historical_sampled',2.5),
    ]
    checks = []
    for iid, variant, ceiling in tqdm(calibration_spec,desc='Calibración autoritativa'):
        selected = old_df[(old_df.id==iid)&(old_df.variant==variant)]
        assert len(selected)==1
        parsed,raw,in_tok,out_tok = judge_local(GOLD_LOOKUP[iid],selected.iloc[0].candidate)
        score = aggregate_v2(GOLD_LOOKUP[iid],parsed)['score_fom5_v2']
        checks.append({'id':iid,'variant':variant,'score':score,'maximum':ceiling,'passes':score<=ceiling})
    calibration = pd.DataFrame(checks)
    calibration.to_csv(new_dir/'calibration_new_judge.csv',index=False,encoding='utf-8-sig')
    display(calibration)
    assert calibration.passes.all(), 'Calibración fallida; detener antes del benchmark.'
    scored = {}
    for variant in ('baseline_direct_new_run','qlora_new_run'):
        source = [json.loads(line) for line in (new_dir/f'{variant}.jsonl').read_text(encoding='utf-8').splitlines()]
        source_by_id = {row['id']:row for row in source}
        assert len(source_by_id)==30 and set(source_by_id)=={row['id'] for row in benchmark}
        checkpoint = new_dir/f'{variant}_judgments_checkpoint.jsonl'
        rows = [json.loads(line) for line in checkpoint.read_text(encoding='utf-8').splitlines()] if checkpoint.exists() else []
        assert len(rows)==len({row['id'] for row in rows})
        done = {row['id'] for row in rows}
        for item in tqdm(benchmark,desc='FOM-5 v2 '+variant):
            if item['id'] in done: continue
            rec = source_by_id[item['id']].copy()
            assert rec['problem_sha256']==hashlib.sha256(item['problem'].encode()).hexdigest()
            started=time.perf_counter()
            parsed,raw,in_tok,out_tok=judge_local(GOLD_LOOKUP[rec['id']],rec['candidate'])
            rec.update(aggregate_v2(GOLD_LOOKUP[rec['id']],parsed))
            rec.update(judge_v2_raw=raw,judge_model=JUDGE_MODEL,
                judge_input_tokens=in_tok,judge_output_tokens=out_tok,
                judge_latency_sec=round(time.perf_counter()-started,3),
                fatal_errors_v2=json.dumps(parsed.get('fatal_errors',[]),ensure_ascii=False))
            with checkpoint.open('a',encoding='utf-8') as stream:
                stream.write(json.dumps(rec,ensure_ascii=False)+'\n')
            rows.append(rec); done.add(rec['id'])
        assert len(rows)==30
        scored[variant] = pd.DataFrame(rows).set_index('id').sort_index()
        scored[variant].reset_index().to_csv(new_dir/f'{variant}_scored.csv',index=False,encoding='utf-8-sig')
    comparison = pd.DataFrame({'baseline':scored['baseline_direct_new_run'].score_fom5_v2,
                               'QLoRA':scored['qlora_new_run'].score_fom5_v2})
    comparison['cambio'] = comparison.QLoRA-comparison.baseline
    comparison.to_csv(new_dir/'comparison_new_30.csv',encoding='utf-8-sig')
    display(comparison)
    print('Medias NUEVAS (no sustituir las oficiales):',comparison[['baseline','QLoRA','cambio']].mean().to_dict())
    print('Casos nuevos: mejoran',int((comparison.cambio>0).sum()),
          'empatan',int((comparison.cambio==0).sum()),
          'empeoran',int((comparison.cambio<0).sum()))
    ci = np.quantile(np.random.default_rng(42).choice(comparison.cambio.to_numpy(),
                       size=(10000,30),replace=True).mean(axis=1),[.025,.975])
    print('IC bootstrap pareado 95% de esta nueva corrida:',ci)


## 7. Reproducir el ajuste completo (opcional, ~16 min en T4)

Esta celda vuelve a entrenar desde el modelo base con las **1050** instancias descargadas del repositorio, una época y los hiperparámetros congelados. Ejecútala solo si quieres repetir el experimento: consume GPU y crea una corrida nueva en el entorno de Colab. La corrida medida original, sus manifiestos y el evaluador autoritativo se preservan en `evidence/` y `src/`. Una corrida nueva requiere generar 31 respuestas y evaluarlas de nuevo; no se le deben asignar los puntajes archivados.


In [ ]:
RUN_FULL_TRAINING = False  # Cambiar a True para una reproducción nueva.
if RUN_FULL_TRAINING:
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                              Trainer, TrainingArguments, set_seed)
    from datasets import Dataset
    assert torch.cuda.is_available()
    set_seed(42)
    model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
    revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
    tok = AutoTokenizer.from_pretrained(model_id, revision=revision)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    def tokenize_row(row):
        prefix = tok.apply_chat_template(row['messages'][:1], tokenize=False, add_generation_prompt=True)
        full = tok.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids']
        ids = tok(full, add_special_tokens=False)['input_ids']
        assert ids[:len(prefix_ids)] == prefix_ids and len(ids) <= 1024
        labels = [-100]*len(prefix_ids) + ids[len(prefix_ids):]
        assert any(value != -100 for value in labels)
        return {'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': labels}
    records = [tokenize_row(row) for row in train]
    dataset = Dataset.from_list(records)
    def collate(batch):
        width = max(len(row['input_ids']) for row in batch)
        return {name: torch.tensor([row[name] + [fill]*(width-len(row[name])) for row in batch])
                for name, fill in [('input_ids', tok.pad_token_id), ('attention_mask', 0), ('labels', -100)]}
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision,
        quantization_config=quant, torch_dtype=torch.float16, device_map={'': 0}, attn_implementation='sdpa')
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=.05,
        bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    fresh = ROOT/'reproduction'
    args = TrainingArguments(output_dir=str(fresh/'checkpoints'), num_train_epochs=1,
        per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=2e-4,
        warmup_ratio=.05, lr_scheduler_type='cosine', fp16=True, bf16=False,
        optim='paged_adamw_8bit', logging_steps=5, save_strategy='steps',
        save_steps=25, save_total_limit=3, report_to='none', remove_unused_columns=False,
        seed=42, data_seed=42, dataloader_num_workers=0, label_names=['labels'])
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collate)
    trainer.train()
    (fresh/'final_adapter').mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(fresh/'final_adapter')
    tok.save_pretrained(fresh/'final_adapter')
    print('Adapter nuevo:', fresh/'final_adapter')


## 8. Auditoría y evaluación posterior

En `evidence/` están el CSV original del baseline, la generación final sin referencias, el CSV completo del juez, manifiestos y el notebook del **evaluador FOM-5 v2 autoritativo**. La sección 6 incorpora su prompt, parser, calibración y agregación para juzgar las generaciones nuevas. `src/` conserva los notebooks de investigación originales. El script complementario `data/build_datasets.py` reconstruye los 30 y 1050 desde fuentes congeladas; no afirma recrear el proceso original de autoría sintética. `src/build_outputs.py` reconstruye los dos CSV oficiales sin volver a generar ni a juzgar. El IC bootstrap de la celda 2 remuestrea casos emparejados con semilla 42 y 10 000 réplicas; no mide variación entre entrenamientos o jueces.
